In [ ]:
# Kaggle has torch/torchvision pre-installed; install the rest quietly
!pip install -q \
    "ultralytics>=8.3.0" \
    "sahi>=0.11.0" \
    "huggingface_hub>=0.24.0" \
    "opencv-python>=4.9.0" \
    "numpy>=1.26.0" \
    "matplotlib>=3.8.0" \
    "pyyaml>=6.0" \
    "requests>=2.31.0" \
    "networkx>=3.2"

In [ ]:
import os

REPO = "https://github.com/MalharRane/P-IDetect.git"
ROOT = "/kaggle/working/PIDetect"

if not os.path.exists(ROOT):
    !git clone {REPO} {ROOT}
else:
    !git -C {ROOT} pull --ff-only

os.chdir(ROOT)
print("CWD:", os.getcwd())


In [ ]:
import sys
sys.path.insert(0, f"{ROOT}/src")

# Smoke-test the import
import pidetect
print("pidetect importable OK")

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    print("VRAM:  ", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

In [ ]:
!python scripts/build_dataset.py \
    --tile    320 \
    --synth-n 200 \
    --seed    42
# Writes:
#   data/tiled_320/          real tiles at 320 px
#   data/synthetic_tiled_320/ synth tiles at 320 px
#   data/merged_320/         merged training set
#   configs/yolo_320.yaml    dataset yaml pointing at merged_320

In [ ]:
from pathlib import Path

for split in ("train", "val", "test"):
    n = len(list(Path(f"data/merged_320/images/{split}").glob("*.jpg")))
    print(f"  {split:<5}: {n:>7,} tiles")

In [ ]:
import os
from pathlib import Path

pts = []
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.endswith(".pt"):
            pts.append(os.path.join(root, f))
            print(os.path.join(root, f))

if pts:
    CHECKPOINT = pts[0]
    print()
    print("Using checkpoint:", CHECKPOINT)
else:
    CHECKPOINT = "yolo11s.pt"
    print("No .pt found in /kaggle/input -- falling back to yolo11s.pt")


In [ ]:
# Confirm checkpoint before training
print("Checkpoint:", CHECKPOINT)
if CHECKPOINT != "yolo11s.pt":
    assert Path(CHECKPOINT).exists(), f"Checkpoint not found: {CHECKPOINT}"
    print("Fine-tuning from existing weights.")
else:
    print("Training from Ultralytics pretrained weights.")


In [ ]:
%env PYTHONPATH=/kaggle/working/PIDetect/src

In [ ]:
!PYTHONPATH=/kaggle/working/PIDetect/src python -m pidetect.detect.train \
    --data   configs/yolo_320.yaml \
    --model  {CHECKPOINT} \
    --aug    small_objects \
    --imgsz  640 \
    --epochs 25 \
    --batch  32 \
    --device 0


In [ ]:
import shutil
from pathlib import Path

# Ultralytics may create train_small_objects, train_small_objects2, etc.
candidates = sorted(Path("runs/detect").glob("train_small_objects*/weights/best.pt"))
if not candidates:
    print("ERROR: no weights found -- check training logs above")
else:
    best = candidates[-1]
    last = best.with_name("last.pt")
    shutil.copy(best, "/kaggle/working/best_320tiles.pt")
    if last.exists():
        shutil.copy(last, "/kaggle/working/last_320tiles.pt")
    sz = Path("/kaggle/working/best_320tiles.pt").stat().st_size / 1e6
    print("Saved from:", best.parent)
    print(f"  best.pt -> /kaggle/working/best_320tiles.pt  ({sz:.1f} MB)")
    if last.exists():
        print("  last.pt -> /kaggle/working/last_320tiles.pt")
